# Symbol Dataset — Augmentation Pipeline

## Context

This notebook prepares the training and validation datasets used to train a **Convolutional Neural Network (CNN)** for symbol recognition.

CNNs are data-hungry: a rule of thumb for image classification is to have at least **500–1 000 labelled examples per class**, with more being better for generalisation. We have **15 symbol classes**, so we target:

| Split | Per-class target | Total |
|---|---|---|
| **Train** | ~667 | **10 000** |
| **Validation** | ~222 | **3 333** |

### Why 10 000 training images?

10 000 images (≈ 667 per class) sits in the sweet spot for a CNN of this complexity:
- Below ~500/class, CNNs overfit quickly — the model memorises the training set rather than learning the underlying shape.
- Above ~1 000/class, gains diminish rapidly for simple binary symbols, and augmentation artefacts start to dominate.
- 10 000 also gives a round budget that is easy to reason about and reproduce.

### Why a 75 / 25 train-validation split?

A naive 90/10 split would yield only ~50 validation images per class, making the validation accuracy **highly volatile**: a single misclassified batch could swing the metric by several percent, which makes hyper-parameter tuning unreliable. A **25 % validation set (~222 images/class)** gives a stable, low-variance estimate of generalisation performance while still leaving 75 % of the data for training — sufficient given that we are augmenting from a much smaller original set.

### What augmentation does

The original dataset has very few images per class. Augmentation synthetically expands it by applying **random rotation (0–360°)** and **small random translations (±10 px)** to existing images, then **re-binarizing** to preserve the hard black/white look of the originals (bilinear interpolation otherwise produces grey anti-aliasing artefacts). Only the *missing* images needed to reach the target are generated — originals are never duplicated or altered.

---

**Files used by this notebook**
| File | Role |
|---|---|
| `augmentation_utils.py` | All pipeline functions (I/O, splitting, augmentation, …) |
| `notebook_utils.py` | Visualisation helpers (bar plots, count summaries) |

## 0 — Imports

In [ ]:
import os
import matplotlib.pyplot as plt
from PIL import Image

from augmentation_utils import (
    load_json,
    save_json,
    count_files,
    compute_targets,
    group_by_class,
    stratified_split,
    copy_images,
    next_available_index,
    complete_split,
    prepare_output_dir,
    augment_image,
)

from notebook_utils import (
    plot_distribution,
    plot_distributions_grid,
    print_split_counts,
)

## 1 — Configuration

The two knobs below drive every downstream computation. Change them here; targets are derived automatically — no hardcoded numbers elsewhere.

| Parameter | Value | Meaning |
|---|---|---|
| `TOTAL_TRAIN` | 10 000 | Total training images across all classes |
| `VAL_RATIO` | 0.25 | Fraction of the full dataset kept for validation |

In [ ]:
DATA_DIR   = "symbols_dataset"       # original images + labels.json
OUT_DIR    = "augm_symbols_dataset"  # output folder (created if missing)

TOTAL_TRAIN = 10_000   # desired total training images
VAL_RATIO   = 0.25     # 25 % of the full dataset → validation

## 2 — Load original labels & compute targets

`compute_targets` uses the formula:

```
train_per_class = TOTAL_TRAIN // num_classes
val_per_class   = train_per_class × VAL_RATIO / (1 − VAL_RATIO)
```

This ensures the ratio is exact regardless of `TOTAL_TRAIN` or the number of classes.

In [ ]:
labels      = load_json(os.path.join(DATA_DIR, "labels.json"))
num_classes = len(set(labels.values()))

TRAIN_TARGET, VAL_TARGET = compute_targets(
    num_classes=num_classes,
    total_train=TOTAL_TRAIN,
    val_ratio=VAL_RATIO,
)

print(f"Classes detected : {num_classes}")
print(f"Original images  : {len(labels)}")
print()
print(f"Train target  : {TRAIN_TARGET:>5} / class  →  {TRAIN_TARGET * num_classes:>6} total")
print(f"Val   target  : {VAL_TARGET:>5} / class  →  {VAL_TARGET   * num_classes:>6} total")
print(f"Effective val ratio : {VAL_TARGET / (TRAIN_TARGET + VAL_TARGET):.1%}")

## 3 — Original dataset distribution

Bar chart of the raw symbol counts before any splitting or augmentation.

In [ ]:
plot_distribution("Original dataset — all classes", labels)
plt.show()

## 4 — Stratified 75 / 25 train / validation split

`stratified_split` shuffles each class independently (with a fixed seed for reproducibility), then assigns the first `VAL_RATIO` fraction to validation and the rest to training. This ensures the class distribution is identical in both splits.

In [ ]:
train_labels, val_labels = stratified_split(labels, val_ratio=VAL_RATIO, seed=42)

print(f"Original total : {len(labels)}")
print(f"→ Train        : {len(train_labels)}")
print(f"→ Validation   : {len(val_labels)}")

print_split_counts(
    train_labels, val_labels,
    TRAIN_TARGET, VAL_TARGET,
    num_classes,
    stage="Before augmentation (originals only)",
)

In [ ]:
plot_distributions_grid([
    ("Train split — original images", train_labels),
    ("Validation split — original images", val_labels),
])

## 5 — Prepare output folder & copy originals

`prepare_output_dir` checks whether the output folder already contains images **and** both JSON split files:
- If yes → returns `False` and loads from disk (no duplication, no re-augmentation).
- If no → creates the folder and returns `True`.
- If partially populated → raises an error (interrupted previous run, needs manual cleanup).

Original images are copied first; only the *missing* images are then generated by augmentation.

In [ ]:
is_fresh_run = prepare_output_dir(OUT_DIR)

if is_fresh_run:
    # Save initial (pre-augmentation) split JSONs
    save_json(train_labels, os.path.join(OUT_DIR, "train.json"))
    save_json(val_labels,   os.path.join(OUT_DIR, "val.json"))

    # Copy original images to output folder
    all_images = list(train_labels.keys()) + list(val_labels.keys())
    copy_images(all_images, src_dir=DATA_DIR, dst_dir=OUT_DIR)

    orig_count, _ = count_files(DATA_DIR)
    out_count,  _ = count_files(OUT_DIR)
    print(f"Original dataset : {orig_count} images")
    print(f"Copied to output : {out_count} images")
else:
    # Folder already complete — reload splits from disk
    train_labels = load_json(os.path.join(OUT_DIR, "train.json"))
    val_labels   = load_json(os.path.join(OUT_DIR, "val.json"))
    print(f"Loaded existing splits — train: {len(train_labels)}, val: {len(val_labels)}")

## 6 — Augmentation examples

One original image alongside 5 augmented variants: random rotation (full 360°) + small translation (±10 px) + re-binarization.

In [ ]:
sample_name = list(train_labels.keys())[0]
sample_img  = Image.open(os.path.join(DATA_DIR, sample_name)).convert("RGB")

fig, axes = plt.subplots(1, 6, figsize=(18, 4))

axes[0].imshow(sample_img, cmap="gray")
axes[0].set_title("Original")
axes[0].axis("off")

for i in range(5):
    aug, angle, tx, ty = augment_image(sample_img)
    axes[i + 1].imshow(aug, cmap="gray")
    axes[i + 1].set_title(f"Aug {i+1}\nθ={angle:.1f}°  dx={tx:.1f} dy={ty:.1f}", fontsize=8)
    axes[i + 1].axis("off")

plt.suptitle(f"{sample_name}  —  label: {train_labels[sample_name]}")
plt.tight_layout()
plt.show()

## 7 — Balance both splits by augmentation

For each class, `complete_split` counts how many images are **already present** in the split and only generates the **missing** ones needed to reach the target. Images that already exist are never duplicated or re-processed.

One progress bar is shown per split (not one per class), so the output stays readable.

In [ ]:
if is_fresh_run:
    start_idx = next_available_index([train_labels, val_labels])

    new_train, start_idx = complete_split(
        split_labels=train_labels,
        target_count=TRAIN_TARGET,
        src_dir=DATA_DIR,
        out_dir=OUT_DIR,
        start_index=start_idx,
        split_name="TRAIN",
    )

    new_val, _ = complete_split(
        split_labels=val_labels,
        target_count=VAL_TARGET,
        src_dir=DATA_DIR,
        out_dir=OUT_DIR,
        start_index=start_idx,
        split_name="VALIDATION",
    )

    # Overwrite JSONs with the final (augmented) label mappings
    save_json(new_train, os.path.join(OUT_DIR, "train.json"))
    save_json(new_val,   os.path.join(OUT_DIR, "val.json"))
    print("Saved final train.json and val.json")
else:
    new_train = train_labels
    new_val   = val_labels
    print("Augmentation skipped — output folder already complete.")

## 8 — Final counts & distribution

Verify that both splits reached their targets (✓) and inspect the per-class bar charts.

In [ ]:
print_split_counts(
    new_train, new_val,
    TRAIN_TARGET, VAL_TARGET,
    num_classes,
    stage="After augmentation",
)

total_out, _ = count_files(OUT_DIR)
print(f"\nTotal image files in '{OUT_DIR}': {total_out}")

In [ ]:
plot_distributions_grid([
    ("Train split — final (after augmentation)", new_train),
    ("Validation split — final (after augmentation)", new_val),
])